# G3 — Patch-Level Faithfulness

**Barrier 2 (Resolution Problem):** Single-pixel perturbations at 256×256 cannot affect
coarse L4 feature maps (16×16). One pixel = 1/256 of one feature activation → effectively zero.

**Fix:** Zero 16×16 blocks aligned to L4's spatial grid. Each block = one feature map location.

**Models tested (block_size=16 for all, L4 alignment):**
- Stage 29 (skip, L3+L4) — pixel Faith=0.069; expect patch Faith to be LOWER (bypass active)
- 9b (no-skip, L3+L4) — pixel Faith=0.035; expect patch Faith HIGHER than pixel
- 9a (no-skip, L4) — pixel Faith=0.012; expect patch Faith MUCH HIGHER than pixel
- 9L3 (no-skip, L3) — pixel Faith=0.060; also test with block_size=8 (L3-aligned)

**Success criteria:**
- 9a patch Faith > pixel Faith (0.012) — validates Barrier 2 is metric/architecture mismatch
- 9b patch Faith > Stage 29 patch Faith — validates Barrier 1 at correct granularity

In [1]:
import sys, os
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(_root)
sys.path.insert(0, _root)
os.environ.setdefault('PYTORCH_MPS_HIGH_WATERMARK_RATIO', '0.0')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path

from src.data.mmwhs_dataset import MMWHSSliceDataset, NUM_CLASSES
from src.models.proto_seg_net import ProtoSegNet
from src.models.proto_seg_net_v2 import ProtoSegNetV2
from src.metrics.patch_faithfulness import patch_faithfulness_patient

DEVICE   = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR = 'data/pack/processed_data'
CKPT_DIR = Path('checkpoints')
OUT_DIR  = Path('results/v10')
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_SLICES = 50   # slices per patient (test set has 484 slices total)

print(f'Device: {DEVICE}')

# Adapter: wrap ProtoSegNet (2-output) to 3-output interface
class Adapter(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, x, **kw):
        out = self.base(x)
        if len(out) == 2:
            logits, hm = out
            n = len(self.base.proto_levels)
            w = torch.full((x.size(0), n), 1.0/n, device=x.device)
            return logits, hm, w
        return out  # already 3-output (ProtoSegNetV2)

Device: mps


In [2]:
# Load test images once
test_ds = MMWHSSliceDataset(DATA_DIR, 'ct', 'test', augment=False, preload=True)
imgs_test = torch.stack([test_ds[i]['image'] for i in range(len(test_ds))])
print(f'Test images: {tuple(imgs_test.shape)}  ({len(test_ds)} slices)')

Test images: (484, 1, 256, 256)  (484 slices)


## Helper: load ProtoSegNet (with skip)

In [3]:
def load_proto_seg_net(ckpt_name, proto_levels):
    ckpt = torch.load(CKPT_DIR / ckpt_name, map_location='cpu', weights_only=False)
    model = ProtoSegNet(
        n_classes=NUM_CLASSES,
        proto_levels=proto_levels,
        use_level_attention=ckpt.get('use_level_attention', False),
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return Adapter(model).to(DEVICE)

def load_proto_seg_net_v2(ckpt_name, proto_levels):
    ckpt = torch.load(CKPT_DIR / ckpt_name, map_location='cpu', weights_only=False)
    model = ProtoSegNetV2(
        n_classes=NUM_CLASSES,
        proto_levels=proto_levels,
        use_attention=ckpt.get('use_attention', False),
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return Adapter(model).to(DEVICE)

## 1. Stage 29 — skip, L3+L4, block_size=16

In [4]:
print('Loading Stage 29 (skip, L3+L4)…')
m29 = load_proto_seg_net('proto_seg_ct_l3l4_warmstart.pth', [3, 4])
print('Running patch faithfulness (block_size=16)…')
pf29 = patch_faithfulness_patient(m29, imgs_test, DEVICE, block_size=16, max_slices=MAX_SLICES)
print(f'  Stage 29 patch Faith (bs=16): {pf29["patch_faithfulness"]:.4f}  (std {pf29["patch_faithfulness_std"]:.4f})')
pd.DataFrame([pf29]).to_csv(OUT_DIR / 'xai_patch_faithfulness_stage29.csv', index=False)
del m29

Loading Stage 29 (skip, L3+L4)…
Running patch faithfulness (block_size=16)…


  Stage 29 patch Faith (bs=16): 0.2122  (std 0.1208)


## 2. 9b — no-skip, L3+L4, block_size=16

In [5]:
print('Loading 9b (no-skip, L3+L4)…')
m9b = load_proto_seg_net_v2('proto_seg_ct_v2_l34.pth', [3, 4])
print('Running patch faithfulness (block_size=16)…')
pf9b = patch_faithfulness_patient(m9b, imgs_test, DEVICE, block_size=16, max_slices=MAX_SLICES)
print(f'  9b patch Faith (bs=16): {pf9b["patch_faithfulness"]:.4f}  (std {pf9b["patch_faithfulness_std"]:.4f})')
pd.DataFrame([pf9b]).to_csv(OUT_DIR / 'xai_patch_faithfulness_9b.csv', index=False)
del m9b

Loading 9b (no-skip, L3+L4)…
Running patch faithfulness (block_size=16)…


  9b patch Faith (bs=16): 0.2003  (std 0.1463)


## 3. 9a — no-skip, L4 only, block_size=16

In [6]:
print('Loading 9a (no-skip, L4)…')
m9a = load_proto_seg_net_v2('proto_seg_ct_v2_l4.pth', [4])
print('Running patch faithfulness (block_size=16)…')
pf9a = patch_faithfulness_patient(m9a, imgs_test, DEVICE, block_size=16, max_slices=MAX_SLICES)
print(f'  9a patch Faith (bs=16): {pf9a["patch_faithfulness"]:.4f}  (std {pf9a["patch_faithfulness_std"]:.4f})')
pd.DataFrame([pf9a]).to_csv(OUT_DIR / 'xai_patch_faithfulness_9a.csv', index=False)
del m9a

Loading 9a (no-skip, L4)…
Running patch faithfulness (block_size=16)…


  9a patch Faith (bs=16): 0.2585  (std 0.0981)


## 4. 9L3 — no-skip, L3 only, block_size=8 (L3-aligned) and block_size=16

In [7]:
print('Loading 9L3 (no-skip, L3)…')
m9l3 = load_proto_seg_net_v2('proto_seg_ct_v2_l3.pth', [3])
print('Running patch faithfulness (block_size=8, L3-aligned)…')
pf9l3_8 = patch_faithfulness_patient(m9l3, imgs_test, DEVICE, block_size=8, max_slices=MAX_SLICES)
print(f'  9L3 patch Faith (bs=8):  {pf9l3_8["patch_faithfulness"]:.4f}  (std {pf9l3_8["patch_faithfulness_std"]:.4f})')
print('Running patch faithfulness (block_size=16, L4-aligned)…')
pf9l3_16 = patch_faithfulness_patient(m9l3, imgs_test, DEVICE, block_size=16, max_slices=MAX_SLICES)
print(f'  9L3 patch Faith (bs=16): {pf9l3_16["patch_faithfulness"]:.4f}  (std {pf9l3_16["patch_faithfulness_std"]:.4f})')
pd.DataFrame([pf9l3_8]).to_csv(OUT_DIR / 'xai_patch_faithfulness_9l3_bs8.csv', index=False)
pd.DataFrame([pf9l3_16]).to_csv(OUT_DIR / 'xai_patch_faithfulness_9l3_bs16.csv', index=False)
del m9l3

Loading 9L3 (no-skip, L3)…
Running patch faithfulness (block_size=8, L3-aligned)…


  9L3 patch Faith (bs=8):  0.2092  (std 0.0724)
Running patch faithfulness (block_size=16, L4-aligned)…


  9L3 patch Faith (bs=16): 0.1587  (std 0.1298)


## 5. Summary Table

In [8]:
pixel_faith = {
    'Stage 29 (skip, L3+L4)': 0.069,
    '9b (no-skip, L3+L4)':    0.035,
    '9a (no-skip, L4)':       0.012,
    '9L3 (no-skip, L3)':      0.060,
}
patch_faith_bs16 = {
    'Stage 29 (skip, L3+L4)': pf29['patch_faithfulness'],
    '9b (no-skip, L3+L4)':    pf9b['patch_faithfulness'],
    '9a (no-skip, L4)':       pf9a['patch_faithfulness'],
    '9L3 (no-skip, L3)':      pf9l3_16['patch_faithfulness'],
}
patch_faith_aligned = {
    'Stage 29 (skip, L3+L4)': pf29['patch_faithfulness'],    # bs=16 = L4 aligned
    '9b (no-skip, L3+L4)':    pf9b['patch_faithfulness'],    # bs=16 = L4 aligned
    '9a (no-skip, L4)':       pf9a['patch_faithfulness'],    # bs=16 = L4 aligned
    '9L3 (no-skip, L3)':      pf9l3_8['patch_faithfulness'], # bs=8  = L3 aligned
}

rows = []
for name in pixel_faith:
    pf_px = pixel_faith[name]
    pf_16 = patch_faith_bs16[name]
    pf_al = patch_faith_aligned[name]
    rows.append({'Model': name,
                 'Pixel Faith': pf_px,
                 'Patch Faith (bs=16)': pf_16,
                 'Patch Faith (aligned)': pf_al,
                 'Ratio (patch/pixel)': pf_16 / pf_px if pf_px > 0 else float('nan')})

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / 'xai_patch_faithfulness_summary.csv', index=False)

print('='*80)
print('  Patch-Level vs Pixel-Level Faithfulness')
print('='*80)
print(df.round(4).to_string(index=False))
print()
print('Barrier 2 check: 9a patch Faith >> 9a pixel Faith?')
print(f'  9a pixel={pixel_faith["9a (no-skip, L4)"]:.3f}  patch={patch_faith_bs16["9a (no-skip, L4)"]:.3f}')
print()
print('Barrier 1 check at patch level: 9b patch Faith > Stage 29 patch Faith?')
print(f'  9b={patch_faith_bs16["9b (no-skip, L3+L4)"]:.3f}  Stage29={patch_faith_bs16["Stage 29 (skip, L3+L4)"]:.3f}')


  Patch-Level vs Pixel-Level Faithfulness
                 Model  Pixel Faith  Patch Faith (bs=16)  Patch Faith (aligned)  Ratio (patch/pixel)
Stage 29 (skip, L3+L4)        0.069               0.2122                 0.2122               3.0756
   9b (no-skip, L3+L4)        0.035               0.2003                 0.2003               5.7217
      9a (no-skip, L4)        0.012               0.2585                 0.2585              21.5418
     9L3 (no-skip, L3)        0.060               0.1587                 0.2092               2.6443

Barrier 2 check: 9a patch Faith >> 9a pixel Faith?
  9a pixel=0.012  patch=0.259

Barrier 1 check at patch level: 9b patch Faith > Stage 29 patch Faith?
  9b=0.200  Stage29=0.212
